**En aquesta tasca t’enfrontaràs a un exercici de neteja i analítica de dades.**

Tenim un dataset provinent d’una enquesta als nostres treballadors i treballadors, i hem de garantir que les dades es processes correctament, tant en format com en llegibilitat.


___

In [1]:
import pandas as pd
from datetime import date as fechas
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import pprint

# Nivell 1

## 1.1.
Importa com un DataFrame l'arxiu sprint10.xlsx. Assegura't que el fitxer s'importa correctament, amb els noms de columnes que li corresponen, sense manipular l'arxiu original.

Ordena el DataFrame pel país d'origen. En cas d'empat, ordena pel nom de la ciutat.

Mostra les primeres 10 files.

In [2]:

encuesta_limpia = pd.read_excel(r"sprint10.xlsx", header= 2)

encuesta_limpia.columns = encuesta_limpia.iloc[0]
encuesta_limpia = encuesta_limpia.iloc[1:].reset_index(drop= True)
encuesta_limpia.index.name = 'ID'
encuesta_limpia = encuesta_limpia.loc[:, encuesta_limpia.columns.notna()]
encuesta_limpia.columns.name = None
encuesta_limpia['Dia de Naixement'] = pd.to_numeric(encuesta_limpia['Dia de Naixement'])
encuesta_limpia['Mes de Naixement'] = pd.to_numeric(encuesta_limpia['Mes de Naixement'])
encuesta_limpia['Any de Naixement'] = pd.to_numeric(encuesta_limpia['Any de Naixement'])
encuesta_limpia['Salari mensual'] = (encuesta_limpia['Salari mensual']
                                     .str.replace('€', '', regex=False)
                                     .str.replace('.', '', regex=False)
                                     .str.strip()
)
encuesta_limpia['Salari mensual'] = pd.to_numeric(encuesta_limpia['Salari mensual'],
                                                   errors='coerce')
encuesta_limpia = encuesta_limpia.rename(columns={'Salari mensual': 'Salari mensual €'})

encuesta_df = encuesta_limpia.copy()

encuesta_df_ordenada = encuesta_df.sort_values(ascending=True, by=["País d'origen", "Ciutat"])
encuesta_df_ordenada.head(10)



,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual €,Fills,No Fills,Grup Professional
ID,,,,,,,,,,,,,
21,Mia,Schneider Fischer,28973553Z,Alemanya,Berlín,22,10,1976,A,951,NaN,True,Grup A
154,Laura,Schneider Fischer,37399141L,Alemanya,Berlín,2,2,1958,D,1769,True,NaN,Grup B
224,Lea,Schneider Schneider,37368317L,Alemanya,Berlín,23,10,2005,D,2013,NaN,True,Grup B
278,Mia,Fischer,21390098Z,Alemanya,Berlín,11,8,1950,D,1557,True,NaN,Grup B
602,Jonas,Schneider,44060014R,Alemanya,Berlín,22,11,1985,H,2754,True,NaN,Grup D
871,Lea,Fischer,14773153R,Alemanya,Berlín,9,9,1986,D,1370,True,NaN,Grup A
281,Lea,Müller,23266650S,Alemanya,Hamburg,14,4,2003,D,1314,NaN,True,Grup A
435,Anna,Müller,83274277X,Alemanya,Hamburg,1,1,1987,D,2464,NaN,True,Grup C
444,Laura,Schmidt Müller,60161784X,Alemanya,Hamburg,15,6,1987,NC,2035,True,NaN,Grup C


Addicionalment, fes un print on comprovi que el DNI només té valors únics.

In [3]:
if encuesta_df['DNI'].is_unique:
    print("La columna DNI tiene solo valores únicos.")
else:
    print("La columna DNI tiene valores repetidos.")

La columna DNI tiene solo valores únicos.


## 1.2.
Crea una columna que sigui el nom complet.

In [4]:
encuesta_df['Nom complet'] = encuesta_df.Nom.str.cat(encuesta_df.Cognoms, sep=' ')

Crea una columna si la persona és nascuda a Espanya o no.


In [5]:
def origen_espanya(pais: str) -> str:
    """
    Evalúa si el valor de la columna es igual o no a 'Espanya'.

    Args:
        pais: str, el valor de cada celda a evaluar

    Returns:
        'Indeterminado'     si el valor es nulo o no str.
        'Sí'                si el valor es Espanya.
        'No'                en el resto de casos.
    """
    if pd.isna(pais) or not isinstance(pais, str):
        return 'Indeterminado'
    elif pais == 'Espanya':
        return 'Sí'
    else:
        return 'No'
    
encuesta_df['Origen Espanya'] = encuesta_df["País d'origen"].apply(origen_espanya)

Posa el DNI com a índex del DataFrame (noms de files).

In [6]:
encuesta_df = encuesta_df.reset_index().set_index('DNI')

Substitueix el nom de les columnes Dia de Naixement, Mes de Naixement i Any de Naixement per Dia, Mes i Any.

In [7]:
encuesta_df = encuesta_df.rename(columns={'Dia de Naixement': 'Dia', 
                                          'Mes de Naixement': 'Mes', 
                                          "Any de Naixement": 'Any'})

Substitueix H per Home, D per Dona, A per Altres i NC per una dada faltant (nan/null/na).

In [8]:
encuesta_df['Gènere'] = encuesta_df['Gènere'].replace({
    'H': 'Home',
    'D': 'Dona',
    'A': 'Altres',
    'NC': pd.NA
})

Mostra tots els canvis que has realitzat en una sola taula.

In [9]:
encuesta_df

,ID,Nom,Cognoms,País d'origen,Ciutat,Dia,Mes,Any,Gènere,Salari mensual €,Fills,No Fills,Grup Professional,Nom complet,Origen Espanya
DNI,,,,,,,,,,,,,,,
16928694K,0,Inês,Ferreira Silva,Portugal,Lisboa,25,2,1953,Dona,1144,NaN,True,Grup B,Inês Ferreira Silva,No
27724652S,1,Clara,Sánchez Martínez,Espanya,Barcelona,18,3,1996,Dona,1253,True,NaN,Grup A,Clara Sánchez Martínez,Sí
38141675A,2,Fatima,Fassi,Marroc,Rabat,6,11,2005,Altres,1441,True,NaN,Grup A,Fatima Fassi,No
59157262R,3,Khadija,Bennani Bennani,Marroc,Rabat,20,1,1995,Dona,1944,NaN,True,Grup B,Khadija Bennani Bennani,No
69630528M,4,Toni,Sánchez García,Espanya,Barcelona,9,8,1999,Home,1043,NaN,True,Grup A,Toni Sánchez García,Sí
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25161375F,995,Marta,Ferrer Ferrer,Espanya,Sevilla,1,6,1951,Dona,1216,NaN,True,Grup B,Marta Ferrer Ferrer,Sí
52145541P,996,Joan,García,Espanya,Sevilla,11,4,1959,Home,971,NaN,True,Grup A,Joan García,Sí
69760120X,997,Laia,Ferrer Martínez,Espanya,Barcelona,11,11,1980,Dona,682,NaN,True,Grup A,Laia Ferrer Martínez,Sí


## 1.3.
Junta les columnes Fills i No Fills en una sola columna, utilitzant el mètode .apply() i definint una funció que resolgui el problema. La columna nova ha de dir-se "Fills" i prendre els valors "Sí" o "No".

In [10]:
def evaluador_true_false(fila: pd.Series, columna1: str, columna2: str) -> str:
    """
    Evalua los valores booleanos de dos columnas en una fila y devuelve una etiqueta.

    Args:
        fila: pd.Series del dataframe, recibida por el apply(axis=1).
        columna1: str, nombre de la columna que representa el valor positivo.
        columna2: str, nombre de la columna que representa el valor negativo.

    Returns:
        str:
            'Sí'            para el valor True
            'No'            para el valor False
            'Indeterminado' si ambas columnas son NaN
            'Error'         cualquier otro caso inesperado 
    """
    if fila[columna1] == True:
        return 'Sí'
    elif fila[columna2] == True:
        return 'No'
    elif pd.isna(fila[columna1]) and pd.isna(fila[columna2]):
        return 'Indeterminado'
    else:
        return 'Error'
  
encuesta_df['Fills'] = encuesta_df[['Fills', 'No Fills']].apply(evaluador_true_false, axis=1, args=('Fills', 'No Fills'))
encuesta_df = encuesta_df.drop(columns='No Fills')
encuesta_df

,ID,Nom,Cognoms,País d'origen,Ciutat,Dia,Mes,Any,Gènere,Salari mensual €,Fills,Grup Professional,Nom complet,Origen Espanya
DNI,,,,,,,,,,,,,,
16928694K,0,Inês,Ferreira Silva,Portugal,Lisboa,25,2,1953,Dona,1144,No,Grup B,Inês Ferreira Silva,No
27724652S,1,Clara,Sánchez Martínez,Espanya,Barcelona,18,3,1996,Dona,1253,Sí,Grup A,Clara Sánchez Martínez,Sí
38141675A,2,Fatima,Fassi,Marroc,Rabat,6,11,2005,Altres,1441,Sí,Grup A,Fatima Fassi,No
59157262R,3,Khadija,Bennani Bennani,Marroc,Rabat,20,1,1995,Dona,1944,No,Grup B,Khadija Bennani Bennani,No
69630528M,4,Toni,Sánchez García,Espanya,Barcelona,9,8,1999,Home,1043,No,Grup A,Toni Sánchez García,Sí
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25161375F,995,Marta,Ferrer Ferrer,Espanya,Sevilla,1,6,1951,Dona,1216,No,Grup B,Marta Ferrer Ferrer,Sí
52145541P,996,Joan,García,Espanya,Sevilla,11,4,1959,Home,971,No,Grup A,Joan García,Sí
69760120X,997,Laia,Ferrer Martínez,Espanya,Barcelona,11,11,1980,Dona,682,No,Grup A,Laia Ferrer Martínez,Sí


## 1.4.
Crea una taula resum que permeti veure el sou mig, medià, mínim i màxim per Gènere.

Ordena la taula en funció del sou mig.

In [11]:
resum_salari_per_genere = encuesta_df.groupby(['Gènere'])['Salari mensual €'].agg(['mean', 'median', 'max', 'min']).round(2).sort_values('mean')
resum_salari_per_genere

,mean,median,max,min
Gènere,,,,
Dona,1469.44,1361.5,3021,665
Altres,1626.59,1545.0,3175,703
Home,1643.25,1531.0,3356,737


## 1.5.
Crea una taula resum amb el salari mig per gènere (files) i país d'origen (columnes).

Afegeix-hi les mitjanes als marges de la taula.

In [12]:
resum_salari_genere_pais = encuesta_df.pivot_table(
    values='Salari mensual €',
    index='Gènere',
    columns="País d'origen",
    aggfunc='mean',
    margins=True,
    margins_name='median'
).round(2)
resum_salari_genere_pais

País d'origen,Alemanya,Argentina,Colòmbia,Espanya,França,Itàlia,Marroc,Mèxic,Portugal,Regne Unit,median
Gènere,,,,,,,,,,,
Altres,951.00,1141.00,1030.00,1706.18,NaN,1423.00,1365.00,1372.00,1765.00,1921.00,1626.59
Dona,1804.31,1291.80,1497.75,1460.16,1566.47,1247.18,1405.21,1517.80,1488.55,1489.46,1469.44
Home,2067.43,1583.29,1554.67,1682.11,1389.25,1672.88,1531.00,1625.00,1497.00,1162.56,1643.25
median,1851.38,1463.39,1495.54,1581.21,1462.73,1425.95,1447.33,1558.42,1523.33,1423.56,1560.99


(EXTRA): Aplica format condicional a la taula per veure en un color més intens els valors més elevats

In [13]:
resum_salari_genere_pais.style.format('{:.2f}').background_gradient(cmap='Greens')

País d'origen,Alemanya,Argentina,Colòmbia,Espanya,França,Itàlia,Marroc,Mèxic,Portugal,Regne Unit,median
Gènere,,,,,,,,,,,
Altres,951.00,1141.00,1030.00,1706.18,nan,1423.00,1365.00,1372.00,1765.00,1921.00,1626.59
Dona,1804.31,1291.80,1497.75,1460.16,1566.47,1247.18,1405.21,1517.80,1488.55,1489.46,1469.44
Home,2067.43,1583.29,1554.67,1682.11,1389.25,1672.88,1531.00,1625.00,1497.00,1162.56,1643.25
median,1851.38,1463.39,1495.54,1581.21,1462.73,1425.95,1447.33,1558.42,1523.33,1423.56,1560.99


## 1.6.
Crea una columna nova que sigui la data de naixament en format Datetime a partir de les columnes dia, mes i any. Utilitzant aquesta columna crea una funció que donada una data, et calculi l'edat actual a dia d'avui.

Utilitza la funció que acabes de crear per generar una columna nova al DataFrame amb l'edat actual.

In [14]:
encuesta_df = encuesta_df.rename(columns={'Dia': 'day', 
                                          'Mes': 'month', 
                                          'Any': 'year'})
encuesta_df['Data de naixament'] = pd.to_datetime(encuesta_df[['day', 'month', 'year']])

def calcul_edat(data_naixament: pd.Series) -> int:
    """
    Calcula la edad actual según una fecha de nacimiento dada.

    Args:
        data_naixament: datetime, fecha de nacimiento.
    
    Returns:
        edat: int, resultado de restar la data_naixement menos la fecha actual (avui). 
    """
    avui = fechas.today()
    edat = avui.year - data_naixament.year

    if(avui.month, avui.day) < (data_naixament.month, data_naixament.day):
        edat -= 1
    return edat

encuesta_df['Edat actual'] = encuesta_df['Data de naixament'].apply(calcul_edat)

encuesta_df = encuesta_df.rename(columns={'day':'Dia',
                                          'month': 'Mes', 
                                          'year': 'Any'})


In [15]:
encuesta_df

,ID,Nom,Cognoms,País d'origen,Ciutat,Dia,Mes,Any,Gènere,Salari mensual €,Fills,Grup Professional,Nom complet,Origen Espanya,Data de naixament,Edat actual
DNI,,,,,,,,,,,,,,,,
16928694K,0,Inês,Ferreira Silva,Portugal,Lisboa,25,2,1953,Dona,1144,No,Grup B,Inês Ferreira Silva,No,1953-02-25,73
27724652S,1,Clara,Sánchez Martínez,Espanya,Barcelona,18,3,1996,Dona,1253,Sí,Grup A,Clara Sánchez Martínez,Sí,1996-03-18,30
38141675A,2,Fatima,Fassi,Marroc,Rabat,6,11,2005,Altres,1441,Sí,Grup A,Fatima Fassi,No,2005-11-06,20
59157262R,3,Khadija,Bennani Bennani,Marroc,Rabat,20,1,1995,Dona,1944,No,Grup B,Khadija Bennani Bennani,No,1995-01-20,31
69630528M,4,Toni,Sánchez García,Espanya,Barcelona,9,8,1999,Home,1043,No,Grup A,Toni Sánchez García,Sí,1999-08-09,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25161375F,995,Marta,Ferrer Ferrer,Espanya,Sevilla,1,6,1951,Dona,1216,No,Grup B,Marta Ferrer Ferrer,Sí,1951-06-01,75
52145541P,996,Joan,García,Espanya,Sevilla,11,4,1959,Home,971,No,Grup A,Joan García,Sí,1959-04-11,67
69760120X,997,Laia,Ferrer Martínez,Espanya,Barcelona,11,11,1980,Dona,682,No,Grup A,Laia Ferrer Martínez,Sí,1980-11-11,45


___

# Nivell 2

## 2.1.
Utilitzant el següent DataFrame, adjunta la columna "Increment" al dataframe del nivell anterior.

Actualitza la columna salari en funció dels percentatges que s'adjunten. No modifiquis manualment els increments, escriu codi Python per fer les conversions necessàries.

df_increment = pd.DataFrame({"Grup":["Grup A","Grup B","Grup C", "Grup D" ] , "Increment":

["5%","3,5%","2%","8%"]})


In [16]:
df_increment = pd.DataFrame({"Grup":["Grup A","Grup B","Grup C", "Grup D" ] , "Increment": ["5%","3,5%","2%","8%"]})
df_increment = df_increment.rename(columns={'Grup': 'Grup Professional',
                                            'Increment': 'Increment %'})
df_increment['Increment %'] = (df_increment['Increment %']
                             .str.replace('%', '', regex=False)
                             .str.replace(',','.', regex=False)
                             .str.strip())
df_increment['Increment %'] = pd.to_numeric(df_increment['Increment %'],
                                          errors='coerce')

encuesta_df = encuesta_df.reset_index()
encuesta_df = pd.merge(encuesta_df, df_increment, on= 'Grup Professional', how='left')
encuesta_df = encuesta_df.set_index('DNI')

encuesta_df['Salari mensual €'] = encuesta_df['Salari mensual €'] * (1 + (encuesta_df['Increment %'] / 100))

In [17]:
encuesta_df

,ID,Nom,Cognoms,País d'origen,Ciutat,Dia,Mes,Any,Gènere,Salari mensual €,Fills,Grup Professional,Nom complet,Origen Espanya,Data de naixament,Edat actual,Increment %
DNI,,,,,,,,,,,,,,,,,
16928694K,0,Inês,Ferreira Silva,Portugal,Lisboa,25,2,1953,Dona,1184.04,No,Grup B,Inês Ferreira Silva,No,1953-02-25,73,3.5
27724652S,1,Clara,Sánchez Martínez,Espanya,Barcelona,18,3,1996,Dona,1315.65,Sí,Grup A,Clara Sánchez Martínez,Sí,1996-03-18,30,5.0
38141675A,2,Fatima,Fassi,Marroc,Rabat,6,11,2005,Altres,1513.05,Sí,Grup A,Fatima Fassi,No,2005-11-06,20,5.0
59157262R,3,Khadija,Bennani Bennani,Marroc,Rabat,20,1,1995,Dona,2012.04,No,Grup B,Khadija Bennani Bennani,No,1995-01-20,31,3.5
69630528M,4,Toni,Sánchez García,Espanya,Barcelona,9,8,1999,Home,1095.15,No,Grup A,Toni Sánchez García,Sí,1999-08-09,26,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25161375F,995,Marta,Ferrer Ferrer,Espanya,Sevilla,1,6,1951,Dona,1258.56,No,Grup B,Marta Ferrer Ferrer,Sí,1951-06-01,75,3.5
52145541P,996,Joan,García,Espanya,Sevilla,11,4,1959,Home,1019.55,No,Grup A,Joan García,Sí,1959-04-11,67,5.0
69760120X,997,Laia,Ferrer Martínez,Espanya,Barcelona,11,11,1980,Dona,716.10,No,Grup A,Laia Ferrer Martínez,Sí,1980-11-11,45,5.0


## 2.2.
Utilitzant un bucle, exporta en 4 fitxers (format .xlsx o .csv) les dades de cada Grup Professional.

Per exemple: "dades_GrupA.xlsx" , "dades_GrupB.xlsx" ...

In [18]:
for nom_grup, dades_grup in encuesta_df.groupby('Grup Professional'):
    nombre_archivo = f'dades_{nom_grup}.xlsx'
    dades_grup.to_excel(nombre_archivo)
print('Archivos exportados')


Archivos exportados


Exporta un 5è DataFrame en format .xlsx o .csv que contingui quants treballadors hi ha per cada Grup Professional, quin és el seu sou mig i quina és la seva edat mediana.

In [19]:
df_5 = encuesta_df.groupby(['Grup Professional']).agg({'Grup Professional': 'size',
                                                       'Salari mensual €': 'mean',
                                                       'Edat actual': 'median'
}).rename(columns={'Grup Professional': 'Num. treballadors',
                   'Salari mensual €': 'Salari mig',
                   'Edat actual': 'Edat mediana'
                   }).round(2)

df_5.to_excel('dades_grup_resum.xlsx', sheet_name='Resum per grup', header=True)
print('Archivo exportado')

Archivo exportado


___

# Nivell 3

El nivell 3 d’aquest sprint és totalment diferent a d’altres sprints que has fet fins ara, ja que són exercicis més abstractes que requereixen barallar-s’hi bastant. No continuen amb el mateix dataset dels nivells anteriors, sinó que et plantegen dues situacions noves totalment diferents entre elles.


## 3.1.
Crea una funció que prengui un dataframe com a paràmetre d'entrada.

La funció ha de crear (i exportar) un gràfic automàticament per a cada columna del dataframe. Per exemple:

un histograma/boxplot si la variable és numèrica
unes barres dels valors més freqüents si és categòrica
unes barres dels anys més freqüents si la dada està en format data.
La idea és crear una funció que funcioni per qualsevol dataframe, no només amb el que hem treballat fins ara.

Mostra el resultat de la funció en algun dels datasets d’exemple que conté el paquet seaborn. Per exemple, iris, penguins o titanic.

Tingues en consideració que en el següent sprint treballaràs exclusivament amb gràfics. L’objectiu d’aquest exercici no és crear gràfics molt elaborats, sinó resoldre una necessitat de manera ràpida i automàtica.

In [20]:
taxis_df = sns.load_dataset('taxis')
taxis_df

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6428,2019-03-31 09:51:53,2019-03-31 09:55:27,1,0.75,4.5,1.06,0.0,6.36,green,credit card,East Harlem North,Central Harlem North,Manhattan,Manhattan
6429,2019-03-31 17:38:00,2019-03-31 18:34:23,1,18.74,58.0,0.00,0.0,58.80,green,credit card,Jamaica,East Concourse/Concourse Village,Queens,Bronx
6430,2019-03-23 22:55:18,2019-03-23 23:14:25,1,4.14,16.0,0.00,0.0,17.30,green,cash,Crown Heights North,Bushwick North,Brooklyn,Brooklyn
6431,2019-03-04 10:09:25,2019-03-04 10:14:29,1,1.12,6.0,0.00,0.0,6.80,green,credit card,East New York,East Flatbush/Remsen Village,Brooklyn,Brooklyn


In [ ]:
def comprobar_df(el_dataframe: pd.DataFrame) -> bool:
    """ 
    Valida si el argumento es un dataframe y contiene datos.

    Args:
        el_dataframe: El objeto a validar.

    Raises:
        TypeError: Si el objeto no es un dataframe.
        ValueError: Si el dataframe está vacío.    
    """
    
    if not isinstance (el_dataframe, pd.DataFrame):
        raise TypeError("El argumento debe ser un Dataframe.")
    if el_dataframe.empty:
        raise ValueError("El dataframe no puede estar vacío.")
    return True

def grafico_numerica(columna: pd.Series) -> plt.axes:
    """ 
    Crea un histograma para una columna o serie de pandas.

    Args:
        columna: Serie numérica de pandas.

    Returns:
        Objeto Axes de matplotlib con el gráfico.
    """
    ax = columna.plot.hist(bins=20, title=columna.name)
    ax.figure.tight_layout()
    return ax

def grafico_fecha(columna: pd.Series) -> plt.axes:
    """ 
    Crea un gráfico de barras para una columna o serie de pandas.

    Args:
        columna: Serie de pandas en formato fecha.

    Returns:
        Objeto Axes de matplotlib con el gráfico.
    """
    ax = columna.dt.year.value_counts().sort_index().plot.bar(figsize=(12,5), title=columna.name)
    ax.figure.tight_layout()
    return ax

def grafico_categorica(columna: pd.Series) -> plt.axes:
    """ 
    Crea un gráfico de barras horizontales para una columna o serie de pandas
    con un máximo de 10 barras, de haber más categorías se agrupan bajo la etiqueta de 'Otros'

    Args:
        columna: Serie de pandas con valores de tipo str.

    Returns:
        Objeto Axes de matplotlib con el gráfico, 
    """
    frecuencias = columna.value_counts()

    if len(frecuencias) > 10:
        top_n = frecuencias.head(9)
        otros = pd.Series({'Otros': frecuencias.iloc[9:].sum()})
        datos_grafico = pd.concat([otros, top_n])

    else:
        datos_grafico = frecuencias

    ax = datos_grafico.plot.barh(figsize=(10,5), title=columna.name)
    ax.figure.tight_layout()
    return ax

def exportar_grafico(ax: plt.axes, carpeta: str = 'Resumen_df', nombre_df: str = '') -> None:
    """
    Realiza la exportación de los gráficos generados a una subcarpeta en el proyecto.

    Args:
        ax: el objeto que contiene los elementos de la visualización.
        carpeta: nombre por defecto de la carpeta
        nombre_df: nombre de la subcarpeta
    """
    nombre = ax.get_title()
    ruta_carpeta = Path(carpeta) / nombre_df if nombre_df else Path(carpeta)
    ruta_carpeta.mkdir(parents=True, exist_ok=True)
    ruta_carpeta / f"{nombre}.png"
    ax.get_figure().savefig(ruta_carpeta / f'{nombre}.png')

def generador_graficos(un_df: pd.DataFrame, nombre_df: str) -> None:
    """ 
    Función orquestadora. itera sobre cada columna y devuelve un gráfico según 
    el tipo: numérica, categórica o de fecha.

    Args:
        un_df: pd.DataFrame, del que se tomarán los datos para los gráficos.
        nombre_df: str, nombre de la subcarpeta donde se exportarán los gráficos.
    """
    comprobar_df(un_df)

    for col in un_df:
        if pd.api.types.is_numeric_dtype(un_df[col]):
            ax = grafico_numerica(un_df[col])
        elif pd.api.types.is_datetime64_any_dtype(un_df[col]):
            ax = grafico_fecha(un_df[col])
        elif pd.api.types.is_string_dtype(un_df[col]) or pd.api.types.is_bool_dtype(un_df[col]):
            ax = grafico_categorica(un_df[col])
        else:
            continue
        exportar_grafico(ax, nombre_df= nombre_df)
        plt.close()

lo probamos con nuestro dataset de Seaborn: taxis

In [22]:
generador_graficos(taxis_df,'taxis')

Comprobamos que la función sea reproducible con nuestro Dataframe 'encuesta_df'

In [23]:
generador_graficos(encuesta_df,'encuesta')

## 3.2.
Carrega l'arxiu matriu_distancies.xlsx a pandas, de manera que els noms de files i els noms de columnes siguin els de les ciutats. Borra "Las Palmas de Gran Canaria" i "Palma" perquè poguem fer el trajecte en cotxe.

Font: Mejores Rutas

Ens interessa visitar totes les ciutats principals d'Espanya recorrent la mínima distància possible.

No cal que ho facis de forma òptima, ens interessa que desenvolupis una solució raonable utilitzant les eines que tens actualment.

Per exemple, una aproximació senzilla (que no òptima) seria anant sempre a la ciutat més propera que no haguem visitat encara

Fes una funció que donada la matriu de distàncies i la ciutat d'origen, faci una proposta de ruta que sigui el més curta possible que puguis, retornant una llista amb l'ordre de visita. Dóna també la distància total recorreguda.



(EXTRA) Des de quina ciutat la ruta seria més curta amb l'algoritme plantejat

In [24]:
distancies_df = pd.read_excel(r'matriu_distancies.xlsx')
distancies_df = distancies_df.set_index('Unnamed: 0')
distancies_df = distancies_df.drop(index=['Palma', 'Las Palmas de Gran Canaria'], columns=['Palma', 'Las Palmas de Gran Canaria'])
distancies_df = distancies_df.fillna(0)

In [25]:
def ciudad_inicio(df: pd.DataFrame) -> str:
    """
    Calcula la mejor ciudad desde dónde partir teniendo en cuenta la primera distancia mínima
    y segunda distancia mínima en caso de empate. Se excluye a sí misma en la matriz.

    Args:
        df: pd.DataFrame, matriz con los valores de las distancias.

    Returns:
        candidatas[0]: str, la primera opción con valor mínimo.
    """
    minimos = {}
    for columna in df.columns:
        valores = df[columna].drop(index=columna)
        minimos[columna] = sorted(valores.tolist())
    
    menor_valor = min(v[0] for v in minimos.values())
    candidatas = [c for c, vals in minimos.items() if vals[0] == menor_valor]
    
    while len(candidatas) > 1:
        mejor_candidata = None
        mejor_segunda_distancia = float('inf')
        
        for ciudad in candidatas:
            vecina_cercana = minimos[ciudad][0]
            ciudad_vecina = df[ciudad][df[ciudad] == vecina_cercana].index[0]
            
            ciudades_restantes = [c for c in df.columns if c != ciudad and c != ciudad_vecina]
            siguiente_distancia = df[ciudad_vecina][ciudades_restantes].min()
            
            if siguiente_distancia < mejor_segunda_distancia:
                mejor_segunda_distancia = siguiente_distancia
                mejor_candidata = ciudad
        
        candidatas = [mejor_candidata]
    
    return candidatas[0]

def viajante_vecino_cercano(df: pd.DataFrame, ciudad_inicio: str) -> tuple[list,int]:
    """
    Calcula la ruta eligiendo siempre la ciudad no visitada más próxima.

    Args:
        df: pd.DataFrame, matriz con los valores de las distancias.
        ciudad_inicio: str, la mejor ciudad como punto de partida.
        
    Returns:
        tuple:
            ruta: list, con las paradas ordenadas.
            distancia_total: int, la suma de la distancia recorrida.
    """
    ciudades_pendientes = list(df.columns)
    ruta = [ciudad_inicio]
    ciudades_pendientes.remove(ciudad_inicio)
    ciudad_actual = ciudad_inicio
    distancia_total = 0

    while ciudades_pendientes:
        distancias_pendientes = df.loc[ciudad_actual, ciudades_pendientes]
        ciudad_siguiente = distancias_pendientes.idxmin()

        ruta.append(ciudad_siguiente)
        distancia_total += distancias_pendientes[ciudad_siguiente]
        ciudades_pendientes.remove(ciudad_siguiente)
        ciudad_actual = ciudad_siguiente
    
    return ruta, distancia_total

def presentacion_ruta(resultado) -> None:
    """
    Realiza el print del resultado de la función.

    Args:
        resultado: tuple, contiene una lista y un int (en ese orden).
    """
    print(f"Ruta óptima: \n{pprint.pformat(resultado[0])} \nTotal distancia recorrida: {resultado[1]}")

def calculador_ruta(df: pd.DataFrame) -> None:
    """
    Función orquestadora. Valida el dataframe reutilizando comprobar_df(),
    establece la ciudad de inicio con ciudad_inicio(), calcula la ruta 
    óptima y la ruta y la distancia total recorrida con viajante_vecino_cercano()
    e imprime los resultados con presentacion_ruta().

    Args:
        df: pd.DataFrame, matriz con los valores de las distancias. 
    """
    if comprobar_df(df):
        c_inicio = ciudad_inicio(df)
        presentacion_ruta(viajante_vecino_cercano(df, c_inicio))



In [26]:
calculador_ruta(distancies_df)

Ruta óptima: 
['Barcelona',
 'Hospitalet de Llobregat',
 'Zaragoza',
 'Valencia',
 'Alicante',
 'Murcia',
 'Córdoba',
 'Sevilla',
 'Málaga',
 'Valladolid',
 'Gijón',
 'Bilbao',
 'Vigo'] 
Total distancia recorrida: 2778.0
